In [ ]:
%%capture
!pip install keras_tuner

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import Input
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import json
import keras_tuner as kt
from tensorflow.keras.optimizers import SGD

In [ ]:
#Carregar os dados do dataset
iris = load_iris()
X = iris.data
y = to_categorical(iris.target, num_classes=3)
target_names = iris.target_names

In [ ]:
#Pre processamento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#Funcao para criar o modulo com hiperparametros variaveis
def construir_modelo(hp):
  model = Sequential()
  #Entrada
  model.add(Input(shape=(4,))) #4 é o numero de coluna do dataset

  units = hp.Int(f'units', min_value=4, max_value=64, step=4)
  model.add(Dense(units, activation='relu'))

  #saida
  model.add(Dense(3,activation='softmax'))

  #Hiperparametros do otimizador
  lr = hp.Choice('learnin_rate', [1e-2,1e-3,1e-4]) #3 valores
  momentum = hp.Float('momentum', 0.0,0.9, step = 0.1) #10 valores

  optimizer = SGD(learning_rate=lr,momentum=momentum)
  model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
#Configurar a busca com o keras tuner
tuner = kt.RandomSearch(
    construir_modelo,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory="kt_iris_tuning",
    project_name="iris_classification"
)

Reloading Tuner from kt_iris_tuning/iris_classification/tuner0.json


In [ ]:
#Executar a busca
tuner.search(X_train, y_train, epochs=100, validation_split=0.2, verbose=0)

# Resultados da melhor combinação
melhor_modelo = tuner.get_best_models(num_models=1)[0]
melhores_hp = tuner.get_best_hyperparameters(1)[0]
print(f'Melhores hiperparâmetros encontrados: ')
print(f' - Neurônios na camada escondida: {melhores_hp.get("units")}')
print(f' - Learning rate: {melhores_hp.get("learning_rate")}')
print(f' - Momentum: {melhores_hp.get("momentum")}')

# Avaliar no conjunto de teste
loss, acc = melhor_modelo.evaluate(X_test, y_test, verbose=0)
print(f'\nAcurácia no conjunto de teste: {acc:.2f}')

Melhores hiperparâmetros encontrados: 
 - Neurônios na camada escondida: 64
 - Learning rate: 0.01
 - Momentum: 0.30000000000000004



Acurácia no conjunto de teste: 0.93


In [ ]:
tuner.results_summary()

Results summary
Results in kt_iris_tuning/iris_classification
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 07 summary
Hyperparameters:
units: 64
learning_rate: 0.01
momentum: 0.30000000000000004
learnin_rate: 0.01
Score: 0.9166666865348816

Trial 03 summary
Hyperparameters:
units: 32
learning_rate: 0.01
momentum: 0.5
learnin_rate: 0.01
Score: 0.9166666865348816

Trial 05 summary
Hyperparameters:
units: 12
learning_rate: 0.01
momentum: 0.6000000000000001
learnin_rate: 0.01
Score: 0.8958333432674408

Trial 02 summary
Hyperparameters:
units: 4
learning_rate: 0.0001
momentum: 0.5
learnin_rate: 0.01
Score: 0.875

Trial 09 summary
Hyperparameters:
units: 40
learning_rate: 0.001
momentum: 0.7000000000000001
learnin_rate: 0.001
Score: 0.8541666567325592

Trial 08 summary
Hyperparameters:
units: 60
learning_rate: 0.001
momentum: 0.1
learnin_rate: 0.001
Score: 0.7291666567325592

Trial 04 summary
Hyperparameters:
units: 32
learning_rate: 0.01
momentum: 0.0
learni